# 🎭 Notebook 04: Gradio Web Demo cho Nhận Diện Biểu Cảm Khuôn Mặt từ Video (FER)

**Môn học:** Công nghệ phần mềm nâng cao  
**Dự án:** FER-Video-Emotion-Recognition (RAVDESS Dataset - 8 Emotion Classes)  
**Người thực hiện phân đoạn Demo & Báo cáo:** Phan Công Thành  

---  
### 📌 Mục tiêu Notebook
1. Tích hợp pipeline tiền xử lý khung hình mặt chuẩn hóa từ video (`data_preprocessing.ipynb`).
2. Khởi tạo mô hình học sâu kết hợp không gian - thời gian `SpatialTemporalFERModel` (ResNet-18 Backbone + Bidirectional LSTM).
3. Nạp trọng số từ checkpoint đã huấn luyện `best_model.pth` và cấu hình `model_config.json`.
4. Dựng ứng dụng web tương tác trực quan bằng **Gradio** hỗ trợ tải video, hiển thị kết quả phân loại biểu cảm và biểu đồ phân bố xác suất cho 8 cảm xúc RAVDESS.

## Cell 1: Kiểm Tra Môi Trường & Import Thư Viện

In [ ]:
import os
import sys
import json
import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
import gradio as gr

print("PyTorch Version:", torch.__version__)
print("OpenCV Version:", cv2.__version__)
print("Gradio Version:", gr.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Compute Device:", device)

## Cell 2: Cấu Hình Đường Dẫn Dự Án & Kết Nối Google Drive (Colab / Local)

In [ ]:
# Tự động phát hiện môi trường Google Colab hoặc Local
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_DIR = '/content/drive/MyDrive/Công nghệ phần mềm nâng cao/Project_FER_Video' if IN_COLAB else '.'
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
MODEL_PATH = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
CONFIG_PATH = os.path.join(CHECKPOINT_DIR, 'model_config.json')

print("Project Directory:", PROJECT_DIR)
print("Model Checkpoint Path:", MODEL_PATH)
print("Model Config Path:", CONFIG_PATH)
print("Checkpoint Exists:", os.path.exists(MODEL_PATH))

## Cell 3: Ánh Xạ Nhãn Cảm Xúc & Định Nghĩa Hàm Tiền Xử Lý Video (Haar Cascade & Uniform Sampling)

In [ ]:
# Danh mục 8 nhãn cảm xúc chuẩn theo RAVDESS (khớp với data_preprocessing.ipynb)
EMOTIONS = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']
EMOTION_MAP = {
    1: 'neutral', 2: 'calm', 3: 'happy', 4: 'sad',
    5: 'angry', 6: 'fearful', 7: 'disgust', 8: 'surprised'
}

# Khởi tạo Face Detector (Haar Cascade)
cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(cascade_path)
if face_cascade.empty():
    raise RuntimeError("Không thể load Haar Cascade face detector!")

def process_video(video_path, sequence_length=16, image_size=(224, 224)):
    """
    Trích xuất và tiền xử lý 16 khung hình đại diện từ video ngắn.
    Áp dụng thuật toán trích xuất mặt Haar Cascade với lề padding 10%.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return None

    # Lấy đều 16 index khung hình theo np.linspace
    frame_indices = np.linspace(0, total_frames - 1, sequence_length, dtype=int)
    frames = []

    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        success, frame = cap.read()
        if not success or frame is None:
            if len(frames) > 0:
                frames.append(frames[-1].copy())
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

        if len(faces) > 0:
            # Chọn khuôn mặt có diện tích lớn nhất
            x, y, w, h = max(faces, key=lambda rect: rect[2] * rect[3])
            pad_x, pad_y = int(0.1 * w), int(0.1 * h)
            x1 = max(0, x - pad_x)
            y1 = max(0, y - pad_y)
            x2 = min(frame.shape[1], x + w + pad_x)
            y2 = min(frame.shape[0], y + h + pad_y)
            face = frame[y1:y2, x1:x2]
        else:
            face = frame

        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
        face = cv2.resize(face, image_size)
        frames.append(face)

    cap.release()
    if len(frames) == 0:
        return None

    while len(frames) < sequence_length:
        frames.append(frames[-1].copy())

    return np.array(frames[:sequence_length], dtype=np.uint8)

def transform_frames_to_tensor(frames):
    """
    Chuyển đổi chuỗi khung hình uint8 sang Tensor float32 và chuẩn hóa ImageNet.
    """
    video_seq = torch.from_numpy(frames).permute(0, 3, 1, 2).float() / 255.0
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    processed = [normalize(video_seq[t]) for t in range(video_seq.size(0))]
    return torch.stack(processed, dim=0).unsqueeze(0)

## Cell 4: Kiến Trúc Mô Hình SpatialTemporalFERModel (CNN Backbone + Bidirectional RNN)

In [ ]:
class SpatialTemporalFERModel(nn.Module):
    """
    Định nghĩa mô hình nhận diện biểu cảm video kết hợp không gian - thời gian.
    Đảm bảo tương thích hoàn toàn với model_training.ipynb và model_evaluation.ipynb.
    """
    def __init__(self, backbone_name='resnet18', rnn_type='LSTM', hidden_dim=256,
                 num_rnn_layers=2, dropout=0.5, num_classes=8):
        super().__init__()
        self.backbone_name = backbone_name
        self.rnn_type = rnn_type

        # 1. Backbone CNN
        if backbone_name == 'resnet18':
            resnet = models.resnet18(weights=None)
            feature_dim = resnet.fc.in_features
            resnet.fc = nn.Identity()
            self.backbone = resnet
        elif backbone_name == 'mobilenet_v2':
            mobilenet = models.mobilenet_v2(weights=None)
            feature_dim = mobilenet.classifier[1].in_features
            mobilenet.classifier = nn.Identity()
            self.backbone = mobilenet
        else:
            raise ValueError(f"Backbone '{backbone_name}' không hợp lệ!")

        # 2. RNN Mạng chuỗi
        if rnn_type == 'LSTM':
            self.rnn = nn.LSTM(input_size=feature_dim, hidden_size=hidden_dim,
                               num_layers=num_rnn_layers, batch_first=True,
                               bidirectional=True, dropout=dropout if num_rnn_layers > 1 else 0)
        elif rnn_type == 'GRU':
            self.rnn = nn.GRU(input_size=feature_dim, hidden_size=hidden_dim,
                              num_layers=num_rnn_layers, batch_first=True,
                              bidirectional=True, dropout=dropout if num_rnn_layers > 1 else 0)
        else:
            raise ValueError(f"RNN '{rnn_type}' không hợp lệ!")

        # 3. Head Phân loại Classifier MLP
        rnn_out_dim = hidden_dim * 2
        self.classifier = nn.Sequential(
            nn.Linear(rnn_out_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        if x.dim() == 5 and x.shape[-1] == 3:
            x = x.permute(0, 1, 4, 2, 3).float() / 255.0
        batch_size, seq_len, C, H, W = x.shape
        c_in = x.reshape(batch_size * seq_len, C, H, W)
        cnn_features = self.backbone(c_in)
        r_in = cnn_features.view(batch_size, seq_len, -1)
        rnn_out, _ = self.rnn(r_in)
        out_feature = torch.mean(rnn_out, dim=1)  # Temporal GAP
        logits = self.classifier(out_feature)
        return logits

## Cell 5: Tải Checkpoint (`best_model.pth`) và Khởi Tạo Trạng Thái Eval

In [ ]:
def load_trained_model(model_path=MODEL_PATH, config_path=CONFIG_PATH, device=device):
    """
    Nạp trọng số mô hình từ file best_model.pth và kiểm tra tính hợp lệ.
    """
    if not os.path.exists(model_path):
        print(f"[Cảnh báo] Chưa tìm thấy file checkpoint tại: {model_path}")
        return None, None, f"File checkpoint không tồn tại: '{model_path}'"

    try:
        checkpoint = torch.load(model_path, map_location=device)
        config = checkpoint.get('config', {})
        if not config and os.path.exists(config_path):
            with open(config_path, 'r') as f:
                config = json.load(f)

        model = SpatialTemporalFERModel(
            backbone_name=config.get('backbone', 'resnet18'),
            rnn_type=config.get('rnn_type', 'LSTM'),
            hidden_dim=config.get('hidden_dim', 256),
            num_rnn_layers=config.get('num_rnn_layers', 2),
            dropout=config.get('dropout', 0.5),
            num_classes=config.get('num_classes', 8)
        )
        model.load_state_dict(checkpoint['model_state_dict'], strict=True)
        model.to(device)
        model.eval()
        val_acc = checkpoint.get('val_acc', 0.0)
        print(f"[Thành công] Đã nạp Best Model (Epoch {checkpoint.get('epoch', '?')}, Val Acc: {val_acc*100:.2f}%)")
        return model, config, None
    except Exception as e:
        return None, None, f"Lỗi nạp checkpoint: {str(e)}"

## Cell 6: Hàm Dự Đoán Biểu Cảm Video (Inference Function)

In [ ]:
def predict_video_emotion(video_path, model_path_str=MODEL_PATH):
    """
    Hàm xử lý chính kết nối giao diện Gradio với Pipeline Dự Đoán FER.
    """
    if video_path is None:
        return "Vui lòng chọn video", "0.0%", {}, "⚠️ Chưa chọn video đầu vào."

    model, config, err_msg = load_trained_model(model_path_str)
    if model is None:
        return (
            "Thiếu Checkpoint",
            "0.0%",
            {},
            f"❌ Lỗi Checkpoint Mô hình: {err_msg}\n\n"
            f"Hướng dẫn: Vui lòng đặt file trọng số 'best_model.pth' vào thư mục 'checkpoints/' trước khi thực hiện dự đoán."
        )

    frames = process_video(video_path, sequence_length=config.get('sequence_length', 16))
    if frames is None or len(frames) == 0:
        return "Lỗi đọc video", "0.0%", {}, f"❌ Không thể trích xuất khung hình từ video: {video_path}"

    input_tensor = transform_frames_to_tensor(frames).to(device)

    with torch.inference_mode():
        logits = model(input_tensor)
        probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()

    top_idx = int(np.argmax(probs))
    top_emotion = EMOTIONS[top_idx].upper()
    confidence_str = f"{probs[top_idx] * 100:.2f}%"
    probs_dict = {EMOTIONS[i].capitalize(): float(probs[i] * 100) for i in range(len(EMOTIONS))}
    status_log = f"✓ Xử lý thành công {len(frames)} khung hình mặt. Biểu cảm nhận diện: {top_emotion} ({confidence_str})."

    return top_emotion, confidence_str, probs_dict, status_log

## Cell 7: Dựng Giao Diện Web App Gradio & Khởi Chạy

In [ ]:
def launch_gradio_app():
    theme = gr.themes.Soft(primary_hue="indigo", secondary_hue="blue")
    with gr.Blocks(theme=theme, title="RAVDESS Video Emotion Recognition") as demo:
        gr.Markdown(
            """
            # 🎭 RAVDESS Video Emotion Recognition Web Demo
            **Môn học:** Công nghệ phần mềm nâng cao | **Thực hiện:** Phan Công Thành  
            Nhận diện cảm xúc khuôn mặt từ đoạn video ngắn bằng mô hình Spatial-Temporal Deep Learning (ResNet-18 + BiLSTM).
            """
        )
        with gr.Accordion("📌 Danh Sách 8 Nhãn Cảm Xúc Bộ Dữ Liệu RAVDESS", open=False):
            gr.Markdown(
                """
                - **Neutral:** Bình thường / Trung tính  
                - **Calm:** Bình tĩnh / Thư thái  
                - **Happy:** Vui vẻ / Hạnh phúc  
                - **Sad:** Buồn rầu / Thất vọng  
                - **Angry:** Tức giận / Phẫn nộ  
                - **Fearful:** E sợ / Lo âu  
                - **Disgust:** Chán ghét / Gê tởm  
                - **Surprised:** Bất ngờ / Ngạc nhiên  
                """
            )
        with gr.Row():
            with gr.Column(scale=1):
                video_input = gr.Video(label="Tải Video Ngắn (.mp4, .avi, .mov)", sources=["upload"])
                ckpt_path_input = gr.Textbox(label="Đường Dẫn Checkpoint (.pth)", value=MODEL_PATH)
                submit_btn = gr.Button("🚀 Dự Đoán Biểu Cảm", variant="primary", size="lg")
            with gr.Column(scale=1):
                status_box = gr.Textbox(label="Trạng Thái Hệ Thống & Nhật Ký Log", value="Hệ thống sẵn sàng.", interactive=False)
                with gr.Row():
                    res_emotion = gr.Textbox(label="Biểu Cảm Dự Đoán", interactive=False)
                    res_conf = gr.Textbox(label="Độ Tin Cậy", interactive=False)
                res_chart = gr.Label(label="Phân Bố Xác Suất 8 Nhãn Cảm Xúc", num_top_classes=8)

        submit_btn.click(
            fn=predict_video_emotion,
            inputs=[video_input, ckpt_path_input],
            outputs=[res_emotion, res_conf, res_chart, status_box]
        )
    return demo

# Để khởi chạy giao diện khi chạy notebook trực tiếp, bỏ comment dòng dưới:
# demo_app = launch_gradio_app()
# demo_app.launch(share=True)

## Cell 8: Hướng Dẫn Vận Hành & Lưu Ý Trình Bày Báo Cáo

1. **Chuẩn bị Checkpoint:** Đảm bảo file `best_model.pth` được lưu tại `checkpoints/best_model.pth` (trên Google Drive hoặc thư mục làm việc cục bộ).
2. **Thực hiện Demo:** Chọn 1 file video ngắn RAVDESS (ví dụ: `01-01-03-01-01-01-01.mp4` đại diện cho nhãn *Happy*) và bấm nút **Dự Đoán Biểu Cảm**.
3. **Kiểm thử Cảnh báo:** Nếu không tìm thấy file checkpoint, ứng dụng sẽ hiển thị thông báo hướng dẫn người dùng tải trọng số thay vì làm treo ứng dụng.